# Część 1 · Apache Spark: przetwarzanie rozproszone na przykładzie plików FASTQ

**Big Data w genomice · OMICS Data Science 2026 · około 110 minut**

| Czas | Temat |
|---|---|
| 15' | Big Data w genomice, architektura Apache Spark |
| 15' | Obiektowa pamięć masowa (kubełki) |
| 10' | Sesja Spark, Spark UI, odczyt danych |
| 15' | Transformacje i akcje: leniwe wykonanie, plan zapytania |
| 30' | Operacje na ramkach danych: projekcja, filtrowanie, kolumny wyliczane, grupowanie |
| 20' | Partycje, shuffle i zysk z równoległości |
| 5' | Zapis wyników i podsumowanie |

Pracujemy na danych trio (matka, ojciec, syn) z warsztatów GATK: odczyty z fragmentu chromosomu 20 (genom referencyjny b37). Te same próbki przetworzymy w części 2 potokiem WDL.

## 1.1 Big Data w genomice

Jedno badanie całego genomu (WGS, pokrycie 30×) to około 100 GB skompresowanych plików FASTQ i podobnej wielkości plik BAM. Projekty populacyjne obejmują dziesiątki i setki tysięcy genomów; sam plik VCF z wariantami gnomAD ma około 0,8 TB. Takich danych nie da się wygodnie przetwarzać na jednej maszynie w jednym wątku.

Mamy dwie drogi:
* **skalowanie w górę** – większa maszyna (więcej rdzeni, RAM, szybszy dysk); proste, ale ograniczone i drogie,
* **skalowanie w poziom** – wiele maszyn, z których każda przetwarza **fragment** danych; wyniki częściowe są potem łączone.

Druga droga wymaga odpowiedzi na trzy pytania: *jak podzielić dane, gdzie wykonać obliczenia, jak zebrać wyniki*. Dziś zobaczymy trzy odpowiedzi:

| | Apache Spark (cz. 1) | WDL + Cromwell (cz. 2) | polars-bio, vepyr (cz. 3) |
|---|---|---|---|
| Jednostka pracy | partycja danych | shard – osobne zadanie (proces) | partycja / porcja danych w jednym procesie |
| Kto dzieli dane | silnik | autor potoku (`scatter`) | silnik (wątki) |
| Typowa skala | klaster | klaster, HPC, chmura | jedna maszyna |

## 1.2 Architektura Apache Spark

[Apache Spark](https://spark.apache.org/) to otwarta platforma do przetwarzania dużych zbiorów danych, z API dla Pythona (PySpark), Scali, Javy, R i SQL. Przetwarzanie odbywa się w dużej mierze w pamięci operacyjnej.

* **Driver** – proces, w którym działa nasz program (tu: kernel notatnika). Buduje plan obliczeń i rozdziela zadania.
* **Executory** – procesy wykonujące zadania na fragmentach danych (**partycjach**). Na klastrze działają na wielu maszynach.
* **Cluster manager** – przydziela zasoby: tryb lokalny, Spark Standalone, YARN (Hadoop) lub **Kubernetes**.

Hierarchia pracy: **akcja** (np. `count()`) uruchamia **job**, job dzieli się na **etapy** (*stages*) w miejscach, gdzie dane trzeba przetasować między partycjami (*shuffle*), a etap na **zadania** (*tasks*) – po jednym na partycję.

Tak wygląda Spark na klastrze Kubernetes (na przykładzie Google Kubernetes Engine): każdy użytkownik ma własny pod z Jupyterem i driverem Sparka, a executory są uruchamiane jako osobne pody na węzłach roboczych:

![Spark na Google Kubernetes Engine](img/gke.png)

Dziś pracujemy w **trybie lokalnym** (`local[*]`): driver i executor działają w jednym procesie JVM w kontenerze, a zadania wykonują wątki – tyle, ile rdzeni ma Twój komputer. **Ten sam kod działa na klastrze** – zmienia się tylko adres `master` w konfiguracji sesji.

## 1.3 Obiektowa pamięć masowa (kubełki)

Duże dane genomowe przechowuje się zwykle nie na dyskach serwerów, lecz w **obiektowej pamięci masowej**: Google Cloud Storage (`gs://`), Amazon S3 (`s3://`), Azure Blob Storage, a w centrach danych także np. Ceph.

* **Projekt** – wszystkie dane należą do projektu, który ma użytkowników, uprawnienia i rozliczenia.
* **Kubełek (bucket)** – kontener na obiekty. Nazwa jest unikalna globalnie (u wszystkich użytkowników usługi). Kubełków nie można zagnieżdżać ani zmieniać im nazwy. Kubełek może być publiczny.
* **Obiekt** – plik wraz z metadanymi (rozmiar, suma kontrolna, data utworzenia). **Obiekty są niemodyfikowalne**: zmiana oznacza zapis nowej wersji całego obiektu (nie ma dopisywania na końcu pliku).
* **Foldery nie istnieją** – są tylko prefiksy nazw obiektów (`bams/mother.bam`), które narzędzia prezentują jak katalogi.
* Obiekt można czytać **fragmentami** (zakres bajtów). Dzięki temu silniki takie jak Spark, Cromwell czy polars-bio czytają dane wprost z kubełka, równolegle i bez kopiowania całego pliku.

Do pracy z Google Cloud Storage służy narzędzie **`gcloud storage`** (starsze narzędzie to `gsutil` – nadal spotkasz je w wielu instrukcjach):

| Operacja | `gcloud storage` | `gsutil` |
|---|---|---|
| lista kubełków / obiektów | `gcloud storage ls gs://kubełek/prefiks/` | `gsutil ls` |
| lista z rozmiarami i datami | `gcloud storage ls -l gs://kubełek/prefiks/` | `gsutil ls -l` |
| łączny rozmiar | `gcloud storage du --summarize --readable-sizes gs://kubełek/prefiks/` | `gsutil du -sh` |
| metadane obiektu | `gcloud storage objects describe gs://kubełek/obiekt` | `gsutil stat` |
| odczyt (fragmentu) obiektu | `gcloud storage cat [--range=OD-DO] gs://kubełek/obiekt` | `gsutil cat [-r OD-DO]` |
| kopiowanie (lokalnie ↔ kubełek, kubełek ↔ kubełek) | `gcloud storage cp ŹRÓDŁO CEL` | `gsutil cp` |
| utworzenie / usunięcie kubełka | `gcloud storage buckets create` / `delete` | `gsutil mb` / `rb` |
| udostępnianie | `gcloud storage buckets add-iam-policy-binding` | `gsutil iam ch` |

Zapis i tworzenie kubełków wymagają konta w chmurze i zalogowania (`gcloud auth login`) – pokaże je prowadzący. Sami **odczytamy publiczny kubełek** `gs://gatk-tutorials`, z którego pochodzą nasze dane. W obrazie warsztatowym `gcloud` działa w trybie anonimowym (zmienna `CLOUDSDK_AUTH_DISABLE_CREDENTIALS=true`), więc publiczne dane czytamy bez logowania.

In [ ]:
! gcloud storage ls gs://gatk-tutorials/workshop_1906/2-germline/

Wiersze zakończone `/` to prefiksy („foldery”), pozostałe to obiekty. Zajrzyjmy do plików BAM – z rozmiarami i datami utworzenia:

In [ ]:
! gcloud storage ls -l gs://gatk-tutorials/workshop_1906/2-germline/bams/

In [ ]:
! gcloud storage objects describe gs://gatk-tutorials/workshop_1906/2-germline/bams/mother.bam

Pole `generation` to numer wersji obiektu: nadpisanie pliku tworzy nową generację, a nie modyfikuje starej. `md5_hash` i `crc32c_hash` to sumy kontrolne – po pobraniu można sprawdzić, czy plik dotarł bez błędów.

Obiekt możemy przeczytać **fragmentami**. Poniżej pobieramy tylko pierwsze 64 bajty pliku BAM o rozmiarze ponad 20 MB (pliki BAM są skompresowane formatem BGZF, który zaczyna się od bajtów `1f 8b`, jak gzip):

In [ ]:
! gcloud storage cat --range=0-63 gs://gatk-tutorials/workshop_1906/2-germline/bams/mother.bam | od -A d -t x1

Na koniec skopiujmy mały obiekt na dysk – indeks sekwencji referencyjnej (`.fai`: nazwa kontigu i jego długość):

In [ ]:
! mkdir -p wyniki && gcloud storage cp gs://gatk-tutorials/workshop_1906/2-germline/ref/ref.fasta.fai wyniki/
! cat wyniki/ref.fasta.fai

<div class="alert alert-block alert-warning">
<b>Zadanie 1.1:</b> Ile miejsca zajmują łącznie obiekty w prefiksie <code>gs://gatk-tutorials/workshop_1906/2-germline/bams/</code>? Który plik BAM jest największy? Użyj <code>gcloud storage du</code> oraz <code>gcloud storage ls -l</code>.
</div>

## 1.4 Sesja Spark

Program w PySpark zaczyna się od utworzenia sesji (`SparkSession`). `local[*]` oznacza tryb lokalny z liczbą wątków równą liczbie dostępnych rdzeni.

In [ ]:
import os
from pyspark.sql import SparkSession
from pyspark.sql import functions as F

spark = (SparkSession.builder
         .master('local[*]')
         .appName('omics-fastq')
         .config('spark.driver.memory', '2g')
         .config('spark.driver.host', '127.0.0.1')
         .config('spark.driver.bindAddress', '127.0.0.1')
         .getOrCreate())
sc = spark.sparkContext
print('Spark', spark.version, '| master:', sc.master, '| równoległość domyślna:', sc.defaultParallelism)
print('Rdzenie widoczne w kontenerze:', os.cpu_count())

**Spark UI** – otwórz w przeglądarce <a href="http://localhost:4040" target="_blank">http://localhost:4040</a> (port 4040 udostępniliśmy przy uruchamianiu kontenera). Zakładki *Jobs*, *Stages*, *Storage* i *SQL / DataFrame* pokazują, co Spark robi pod spodem. Będziemy tam zaglądać.

## 1.5 Odczyt danych: FASTQ a Parquet

Przypomnijmy: rekord FASTQ zajmuje **cztery linie**:

In [ ]:
! head -8 /home/jovyan/work/data/trio/fastq/mother.fq

Spark potrafi czytać pliki tekstowe, ale dzieli je na partycje **po liniach**, w dowolnych miejscach pliku – rekord FASTQ może trafić do dwóch różnych partycji, a linie tracą powiązanie z rekordem. Pliku skompresowanego gzipem nie da się w ogóle podzielić (trzeba go czytać od początku). Dlatego do formatów genomowych potrzebne są specjalne źródła danych – np. biblioteka SeQuiLa (`FASTQDataSource`), rozszerzenie Sparka rozwijane w zespole biodatageeks.

In [ ]:
lines = spark.read.text('/home/jovyan/work/data/trio/fastq/mother.fq')
lines.show(8, truncate=False)
print('Linie:', lines.count(), '| partycje:', lines.rdd.getNumPartitions())

Dziś odczyty mamy przygotowane w formacie **Apache Parquet** – kolumnowym, skompresowanym formacie ze schematem, który Spark (i polars, pandas, DuckDB, BigQuery...) czyta natywnie i dzieli na partycje bez problemu. Każda próbka to jeden plik z kolumnami `sample_id`, `name`, `seq`, `qual`. Porównajmy rozmiary:

In [ ]:
! ls -lh /home/jovyan/work/data/trio/fastq/ /home/jovyan/work/data/trio/fastq_parquet/

In [ ]:
fastq_path = '/home/jovyan/work/data/trio/fastq_parquet'
fastq_all = spark.read.parquet(fastq_path)
type(fastq_all)

In [ ]:
fastq_all.printSchema()

In [ ]:
print('Kolumny:', len(fastq_all.columns), '| wiersze:', fastq_all.count(), '| partycje:', fastq_all.rdd.getNumPartitions())

In [ ]:
fastq_all.show(5)

In [ ]:
fastq_all.show(3, truncate=False)

## 1.6 Transformacje i akcje: leniwe wykonanie

Operacje na ramkach danych Sparka dzielą się na dwie grupy:

* **transformacje** (`select`, `filter`, `withColumn`, `groupBy(...).agg(...)`, `join`, `orderBy`...) – zwracają **nową ramkę** i niczego nie liczą; Spark tylko dopisuje krok do **planu**,
* **akcje** (`count`, `show`, `collect`, `toPandas`, `write`...) – dopiero one uruchamiają obliczenia (job widoczny w Spark UI).

Takie podejście nazywamy **leniwym wykonaniem** (*lazy evaluation*). Dzięki niemu silnik widzi całe zapytanie naraz i może je zoptymalizować, zanim przeczyta choćby bajt danych. Wrócimy do tej idei w części 3 (polars-bio).

In [ ]:
%%time
fastq_mother = fastq_all.filter("sample_id = 'mother'").select('name', 'seq')

`Czy odbył się już odczyt danych z pliku?` Sprawdź w zakładce *Jobs* w Spark UI, czy pojawiło się nowe zadanie.

In [ ]:
%%time
fastq_mother.count()

Plan zapytania pokazuje metoda `explain()`. Spark (optymalizator Catalyst) przekształca plan logiczny w zoptymalizowany i fizyczny. Zwróć uwagę na linię `FileScan parquet` w planie fizycznym:

* `PushedFilters` – warunki przekazane do czytnika Parquet (*predicate pushdown*): fragmenty pliku, które na pewno nie spełniają warunku, nie są czytane,
* `ReadSchema` – czytane są tylko potrzebne kolumny (*projection pushdown*); kolumna `qual` w ogóle nie zostanie odczytana z dysku.

In [ ]:
fastq_mother.explain(True)

Kolejność `select` i `filter` w kodzie nie ma znaczenia dla wykonania – optymalizator i tak ustawi je najkorzystniej. Porównaj plany fizyczne:

In [ ]:
fastq_all.filter('seq LIKE "GCA%"').select('sample_id', 'seq').explain()
fastq_all.select('sample_id', 'seq').filter('seq LIKE "GCA%"').explain()

Każda akcja liczy plan **od początku**, łącznie z odczytem pliku. Jeśli tej samej ramki używamy wielokrotnie, możemy ją zapamiętać w pamięci executorów metodą `cache()` (zakładka *Storage* w Spark UI). `cache()` też jest leniwe – ramka trafi do pamięci przy pierwszej akcji.

In [ ]:
import time

def timed(label, action):
    start = time.perf_counter()
    result = action()
    print(f'{label}: {time.perf_counter() - start:.2f} s')
    return result

gc_reads = fastq_all.filter(F.col('seq').rlike('^[GC]{5}'))
timed('bez cache, 1. akcja', gc_reads.count)
timed('bez cache, 2. akcja', gc_reads.count)
gc_reads.cache()
timed('cache, 1. akcja (wypełnia pamięć)', gc_reads.count)
timed('cache, 2. akcja', gc_reads.count)
gc_reads.unpersist();

<div class="alert alert-block alert-warning">
<b>Zadanie 1.2:</b> Które z poniższych linii uruchomią job Sparka? Najpierw zgadnij, potem sprawdź w Spark UI.
<pre>
a = fastq_all.filter("sample_id = 'son'")
b = a.withColumn('len', F.length('seq'))
b.printSchema()
b.show(3)
c = b.groupBy('len').count()
c.collect()
</pre>
Następnie wyświetl plan zapytania <code>c</code> i znajdź w nim <code>PushedFilters</code> oraz <code>ReadSchema</code>.
</div>

## 1.7 Operacje na ramkach danych

### Projekcja (`select`, `drop`)

In [ ]:
fastq_all.select('sample_id', 'seq').show(5)

`Czy operacja select wpłynęła na oryginalną ramkę fastq_all?` Ramki Sparka są **niemodyfikowalne** – każda transformacja tworzy nową ramkę. Jeśli chcemy użyć wyniku później, zapisujemy go w zmiennej.

In [ ]:
fastq_no_qual = fastq_all.drop('qual')
fastq_no_qual.printSchema()
fastq_all.printSchema()

### Wartości unikalne i sortowanie
Metody można łączyć w łańcuch – każda zwraca nową ramkę.

In [ ]:
fastq_all.select('sample_id').distinct().orderBy('sample_id').show()

In [ ]:
fastq_all.select('sample_id').distinct().orderBy('sample_id', ascending=False).show()

### Filtrowanie
Warunki w `filter()` zapisujemy jak w klauzuli `WHERE` języka SQL: operatory porównania (`=, !=, >, >=, <, <=`), `IN` / `NOT IN`, `LIKE` (`%` – dowolny ciąg znaków), `IS NULL`, spójniki `AND`, `OR`, `NOT`. Można też użyć wyrażeń kolumnowych z modułu `pyspark.sql.functions` (`F`).

In [ ]:
fastq_mother = fastq_all.filter("sample_id = 'mother'")
fastq_mother.filter('seq LIKE "GCA%"').select('sample_id', 'seq').show(5, truncate=False)

In [ ]:
# to samo zapytanie zapisane wyrażeniami kolumnowymi
fastq_mother.filter(F.col('seq').startswith('GCA')).select('sample_id', 'seq').show(5, truncate=False)

<div class="alert alert-block alert-warning">
<b>Zadanie 1.3:</b> Policz, ile odczytów syna (<code>son</code>) ma sekwencję kończącą się na <code>TGG</code>, a ciąg jakości zaczynający się od <code>==</code>.
</div>

### Kolumny wyliczane
Dostępne są funkcje skalarne (np. `length`, `upper`, `round`, `current_date`) i agregujące (`min`, `max`, `avg`, `sum`, `count`). Pełna lista: [Spark SQL functions](https://spark.apache.org/docs/latest/api/sql/index.html).

In [ ]:
extended_fastq = fastq_all.selectExpr('*', 'length(seq) AS len_seq', 'length(qual) AS len_qual')
extended_fastq.show(5)

In [ ]:
(extended_fastq
 .withColumn('imported_on', F.current_date())
 .withColumn('format', F.lit('FASTQ'))
 .show(5))

Zawartość GC odczytu:

In [ ]:
with_gc = fastq_all.withColumn(
    'gc', F.length(F.regexp_replace('seq', '[^GC]', '')) / F.length('seq'))
with_gc.select('sample_id', 'seq', F.round('gc', 3).alias('gc')).show(5, truncate=False)

### Instrukcje warunkowe

In [ ]:
fastq_dates = fastq_all.withColumn(
    'experiment_date',
    F.when(F.col('sample_id') == 'son', F.to_date(F.lit('2018-11-10')))
     .otherwise(F.to_date(F.lit('2019-01-15'))))
fastq_dates.printSchema()

<div class="alert alert-block alert-warning">
<b>Zadanie 1.4:</b> Sprawdź, czy daty eksperymentów w ramce <code>fastq_dates</code> zostały przypisane poprawnie: dla każdej próbki pokaż datę i liczbę odczytów.
</div>

### Grupowanie i agregacja
`groupBy()` nie zwraca ramki danych, tylko obiekt `GroupedData`, na którym wywołujemy agregację.

In [ ]:
type(fastq_all.groupBy('sample_id'))

In [ ]:
(fastq_all.groupBy('sample_id')
 .agg(F.count('*').alias('reads'),
      F.round(F.avg(F.length('seq')), 1).alias('mean_len'))
 .orderBy('reads')
 .show())

<div class="alert alert-block alert-warning">
<b>Zadanie 1.5:</b> Dla każdej próbki policz średnią zawartość GC odczytów (ramka <code>with_gc</code>) oraz odsetek odczytów, w których GC przekracza 0,6. Posortuj malejąco po średniej.
</div>

Spark SQL: ramkę można zarejestrować jako widok i odpytywać zwykłym SQL-em. Plan i wykonanie są dokładnie takie same jak dla API ramek danych.

In [ ]:
fastq_all.createOrReplaceTempView('reads')
spark.sql("""
    SELECT sample_id, count(*) AS reads, round(avg(length(seq)), 1) AS mean_len
    FROM reads
    GROUP BY sample_id
    HAVING count(*) > 1000
    ORDER BY reads DESC
""").show()

## 1.8 Partycje, shuffle i zysk z równoległości

Dane w Sparku są podzielone na **partycje**; każdą partycję przetwarza jedno zadanie (*task*) w jednym wątku executora. Sprawdźmy, ile odczytów trafiło do każdej partycji:

In [ ]:
print('Partycje:', fastq_all.rdd.getNumPartitions())
fastq_all.groupBy(F.spark_partition_id().alias('partition'), 'sample_id').count().orderBy('partition').show(50)

In [ ]:
fastq_8 = fastq_all.repartition(8)
fastq_8.groupBy(F.spark_partition_id().alias('partition')).count().orderBy('partition').show()

Operacje takie jak `filter` czy `withColumn` wykonują się w każdej partycji niezależnie. `groupBy`, `distinct`, `orderBy` czy `join` wymagają natomiast przesłania wierszy o tym samym kluczu do tej samej partycji – to **shuffle**, najdroższa operacja w przetwarzaniu rozproszonym (zapis na dysk, przesył przez sieć). W planie widać go jako `Exchange`, a w Spark UI jako granicę między etapami (*stages*).

In [ ]:
fastq_all.groupBy('sample_id').count().explain()

### Zapytanie obliczeniowe: rozkład jakości wzdłuż odczytu

Policzymy średnią jakość zasady (Phred = kod ASCII − 33) na każdej pozycji odczytu, osobno dla każdej próbki. `posexplode` zamienia każdy odczyt na tyle wierszy, ile ma zasad – z kilkuset tysięcy odczytów robi się kilkadziesiąt milionów wierszy, więc jest co liczyć.

In [ ]:
def quality_by_position(df):
    return (df
            .select('sample_id', F.posexplode(F.split('qual', '')).alias('pos', 'q'))
            .where(F.col('q') != '')
            .groupBy('sample_id', 'pos')
            .agg(F.avg(F.ascii('q') - 33).alias('mean_phred'))
            .orderBy('sample_id', 'pos'))

quality = timed('rozkład jakości', lambda: quality_by_position(fastq_all).toPandas())
quality.head()

In [ ]:
import matplotlib.pyplot as plt

fig, ax = plt.subplots(figsize=(10, 4))
for sample, group in quality.groupby('sample_id'):
    ax.plot(group['pos'] + 1, group['mean_phred'], label=sample)
ax.set_xlabel('Pozycja w odczycie'); ax.set_ylabel('Średnia jakość (Phred)')
ax.legend(); ax.spines[['top', 'right']].set_visible(False)
plt.show()

### Ile dają kolejne wątki?

Uruchomimy to samo zapytanie z 1, 2, 4... wątkami (`local[n]`). Za każdym razem tworzymy nową sesję Sparka. Dane są czytane od nowa, a wynik jest ten sam. Żeby pomiar nie trwał ułamków sekundy, dane powielamy pięciokrotnie (około 1,2 mln odczytów, 180 mln wartości jakości) – przy bardzo małych danych narzut planowania i uruchamiania zadań przesłania zysk z równoległości.

In [ ]:
from functools import reduce
from pyspark.sql import DataFrame

COPIES = 5

def run_with_threads(n):
    global spark
    spark.stop()
    spark = (SparkSession.builder.master(f'local[{n}]').appName(f'omics-fastq-{n}')
             .config('spark.driver.memory', '2g')
             .config('spark.driver.host', '127.0.0.1')
             .config('spark.driver.bindAddress', '127.0.0.1')
             .getOrCreate())
    df = spark.read.parquet(fastq_path)
    df = reduce(DataFrame.unionByName, [df] * COPIES).repartition(max(n, 4) * 2)
    start = time.perf_counter()
    quality_by_position(df).collect()
    return time.perf_counter() - start

cores = os.cpu_count()
threads = sorted({t for t in (1, 2, 4, 8, cores) if t <= cores})
times = {}
for n in threads:
    times[n] = run_with_threads(n)
    print(f'local[{n}]: {times[n]:.1f} s')

In [ ]:
fig, ax = plt.subplots(figsize=(6, 4))
speedup = [times[1] / times[n] for n in threads]
ax.plot(threads, speedup, 'o-', label='zmierzone przyspieszenie')
ax.plot(threads, threads, '--', color='grey', label='idealne (liniowe)')
ax.set_xlabel('Wątki (local[n])'); ax.set_ylabel('Przyspieszenie względem 1 wątku')
ax.set_xticks(threads); ax.legend(); ax.spines[['top', 'right']].set_visible(False)
plt.show()

`Dlaczego przyspieszenie jest mniejsze niż liniowe?` Część pracy jest sekwencyjna (planowanie, odczyt metadanych, scalanie wyników w driverze), shuffle wymaga zapisu i odczytu danych pośrednich, a wątki dzielą tę samą pamięć i dysk. Na laptopie rdzenie bywają też różne (wydajne i energooszczędne).

### Ta sama idea na klastrze

Tę samą zasadę widać na klastrze Kubernetes w Google Cloud: adnotacja programem VEP 1 100 291 wariantów z chromosomu 22 (projekt 1000 Genomów, plik VCF 10,5 GB) była dzielona na partycje i wykonywana równolegle przez executory Sparka (biblioteka SeqTender). Wyniki pomiarów:

In [ ]:
executors = [5, 10, 20]
seconds = [16 * 60 + 55, 8 * 60 + 51, 5 * 60]
fig, ax = plt.subplots(figsize=(6, 4))
ax.plot(executors, [s / 60 for s in seconds], 'o-')
for e, s in zip(executors, seconds):
    ax.annotate(f'{s // 60} min {s % 60:02d} s', (e, s / 60), textcoords='offset points', xytext=(6, 6))
ax.set_xlabel('Liczba executorów (po 2 GB RAM)'); ax.set_ylabel('Czas adnotacji [min]')
ax.set_title('VEP na Sparku, GKE, 1000 Genomes chr22')
ax.spines[['top', 'right']].set_visible(False)
plt.show()

Dwukrotnie więcej executorów (5 → 10) skraca czas niemal dwukrotnie, kolejne podwojenie (10 → 20) – już tylko 1,8 raza. W części 3 zobaczymy, ile tej samej pracy (adnotacji wariantów) da się wykonać na jednym laptopie dzięki narzędziu vepyr.

## 1.9 Zapis wyników: partycjonowany Parquet

Wyniki zapisujemy zwykle w formacie kolumnowym, podzielone według często filtrowanej kolumny. Każda wartość `sample_id` trafi do osobnego katalogu – taki sam układ stosuje się w kubełkach (`gs://.../sample_id=son/...`).

In [ ]:
(spark.read.parquet(fastq_path)
 .withColumn('gc', F.length(F.regexp_replace('seq', '[^GC]', '')) / F.length('seq'))
 .write.mode('overwrite').partitionBy('sample_id')
 .parquet('wyniki/fastq_by_sample'))

In [ ]:
! find wyniki/fastq_by_sample -name '*.parquet' | sort | head

<div class="alert alert-block alert-warning">
<b>Zadanie 1.6:</b> Wczytaj <code>wyniki/fastq_by_sample</code>, wybierz tylko odczyty syna i wyświetl plan zapytania. Znajdź w nim <code>PartitionFilters</code> – ile katalogów (próbek) Spark musi odczytać?
</div>

## Podsumowanie

* Spark dzieli dane na **partycje** i przetwarza je równolegle; o podziale decyduje silnik (i format danych).
* Transformacje są **leniwe**: budują plan, który optymalizator upraszcza (pushdown filtrów i kolumn), a obliczenia uruchamia dopiero akcja.
* `groupBy`, `join`, `orderBy` wymagają **shuffle** – granicy etapów i najdroższej części pracy.
* Dane w **kubełkach** są niemodyfikowalnymi obiektami, które można czytać fragmentami – dlatego silniki czytają je bez kopiowania.

W części 2 o podziale pracy zdecyduje **autor potoku** (`scatter` w WDL), a jednostką pracy będzie całe zadanie – osobny proces z własnym programem (bwa, GATK).

In [ ]:
spark.stop()